In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import pickle
import os

# Load the training data we saved in notebook 1
df = pd.read_csv('../../data/noaa_goes16_raw/training_data.csv')
df['time_tag'] = pd.to_datetime(df['time_tag'])

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nAlert level distribution:")
print(df['alert_level'].value_counts().sort_index())
print("\nFirst few rows:")
df.head()

Dataset shape: (2003, 6)

Columns: ['time_tag', 'proton_flux_10mev', 'speed', 'density', 'kp_index', 'alert_level']

Alert level distribution:
alert_level
0    1767
1     200
2      36
Name: count, dtype: int64

First few rows:


,time_tag,proton_flux_10mev,speed,density,kp_index,alert_level
0,2026-02-19 04:50:00,0.133965,482.5,2.07,3.0,0
1,2026-02-19 04:55:00,0.175265,482.5,2.07,3.0,0
2,2026-02-19 05:00:00,0.194057,450.4,0.78,3.0,0
3,2026-02-19 05:05:00,0.159656,474.9,2.41,3.0,0
4,2026-02-19 05:10:00,0.149097,476.2,2.09,3.0,0


In [2]:
# Step 1: Scale the features to 0-1 range
# Neural networks train much better on normalized data

feature_cols = ['proton_flux_10mev', 'speed', 'density', 'kp_index']

scaler = MinMaxScaler()
df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

# Save the scaler — we'll need it on the Pi to normalize live sensor data
os.makedirs('../../data/models', exist_ok=True)
with open('../../data/models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Scaler saved.")
print("\nScaled data sample:")
print(df_scaled[feature_cols].head())
print("\nScaled ranges (should all be 0-1):")
print(df_scaled[feature_cols].describe().loc[['min','max']])

Scaler saved.

Scaled data sample:
   proton_flux_10mev     speed   density  kp_index
0           0.042477  0.182444  0.069735  0.461894
1           0.121640  0.182444  0.069735  0.461894
2           0.157661  0.090361  0.024756  0.461894
3           0.091721  0.160643  0.081590  0.461894
4           0.071481  0.164372  0.070432  0.461894

Scaled ranges (should all be 0-1):
     proton_flux_10mev  speed  density  kp_index
min                0.0    0.0      0.0       0.0
max                1.0    1.0      1.0       1.0


In [3]:
# Step 2: Create sliding window sequences for LSTM
# Instead of predicting from a single row, we predict from the last N timesteps
# This lets the model learn temporal patterns — "when flux rises like THIS over 30 mins, a spike follows"

WINDOW_SIZE = 12  # 12 timesteps x 5 minutes = 60 minutes of history per prediction

def create_sequences(data, labels, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(labels[i + window_size])
    return np.array(X), np.array(y)

# Extract scaled features and labels
features = df_scaled[feature_cols].values
labels = df_scaled['alert_level'].values

X, y = create_sequences(features, labels, WINDOW_SIZE)

print("X shape:", X.shape)  # Should be (samples, 12, 4)
print("y shape:", y.shape)  # Should be (samples,)
print("\nWhat this means:")
print(f"  {X.shape[0]} training samples")
print(f"  Each sample = {X.shape[1]} timesteps of history")
print(f"  Each timestep has {X.shape[2]} features")
print("\nLabel distribution after windowing:")
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    label = ['NORMAL', 'ELEVATED', 'CRITICAL'][int(u)]
    print(f"  {int(u)} ({label}): {c} samples")

X shape: (1991, 12, 4)
y shape: (1991,)

What this means:
  1991 training samples
  Each sample = 12 timesteps of history
  Each timestep has 4 features

Label distribution after windowing:
  0 (NORMAL): 1755 samples
  1 (ELEVATED): 200 samples
  2 (CRITICAL): 36 samples


In [4]:
# Step 3: Train/test split — keep time order intact (no shuffling!)
# Shuffling time series data causes data leakage

split_idx = int(len(X) * 0.8)  # 80% train, 20% test

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

# Step 4: Compute class weights to handle imbalance (88/10/2 split)
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes.astype(int), weights))

print("\nClass weights:")
for k, v in class_weight_dict.items():
    label = ['NORMAL', 'ELEVATED', 'CRITICAL'][k]
    print(f"  {k} ({label}): {v:.2f}")

# Save processed data
np.save('../../data/models/X_train.npy', X_train)
np.save('../../data/models/X_test.npy', X_test)
np.save('../../data/models/y_train.npy', y_train)
np.save('../../data/models/y_test.npy', y_test)
print("\nAll arrays saved.")

Training samples: 1592
Test samples: 399

Class weights:
  0 (NORMAL): 0.36
  1 (ELEVATED): 6.10
  2 (CRITICAL): 26.53

All arrays saved.
